In [ ]:
"""
ROC comparison from synthesized scores calibrated to observed validation accuracies.

- Sources (observed single-split or NxK means):
  - MLP (raw+DWT combined): val_acc ≈ 0.8311          [from logs]
  - SpectralCNN (raw 256):  val_acc ≈ 0.7378
  - RF (combined):          val_acc ≈ 0.4889
  - SVD Ridge (raw):        val_acc ≈ 0.4222
  - KNN (spectrum_raw):     mean acc ≈ 0.415 (NxK)
- Method: For each model, synthesize scores for a balanced binary setup using
  Gaussian class-conditional scores with separation d, where accuracy at threshold 0
  equals target_acc. Under equal priors and optimal threshold 0, accuracy = Phi(d/2),
  hence d = 2 * Phi^{-1}(target_acc). We then compute ROC, AUC from the synthetic scores.

Outputs:
- results/roc_synthetic_from_experiments.png
- results/roc_metrics.csv
"""

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from scipy.stats import norm

# Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# Observed validation accuracies from experiments/logs
observed = {
    # name                  target_acc,   notes
    "MLP_combined":        0.8311,  # raw(256)+DWT(12) MLP single-split
    "SpectralCNN_raw":     0.7378,  # raw(256) SpectralCNN single-split
    "RF_combined":         0.4889,  # RF on combined features single-split
    "SVD_Ridge_raw":       0.4222,  # SVD (r=16, lam=1e-3) single-split
    "KNN_spectrum_raw":    0.4150,  # NxK mean (spectrum_raw + KNN)
}

# Synthesis settings
N_POS = 5000
N_NEG = 5000
OUT_DIR = Path("results"); OUT_DIR.mkdir(parents=True, exist_ok=True)

def synthesize_scores_from_acc(acc: float, n_pos: int, n_neg: int, rng: np.random.Generator):
    """Synthesize balanced binary labels and scores given target accuracy.

    Assumptions:
    - Class-conditional scores are Gaussian: pos ~ N(+d/2, 1), neg ~ N(-d/2, 1).
    - Under equal priors and threshold 0, accuracy = Phi(d/2) => d = 2 * Phi^{-1}(acc).
    - We return labels (0/1) and continuous scores for ROC computation.

    Args:
        acc: Target validation accuracy in [0, 1].
        n_pos: Number of positive samples.
        n_neg: Number of negative samples.
        rng:  Numpy Generator for reproducibility.

    Returns:
        y_true (array of shape (n_pos+n_neg,), {0,1}),
        scores (array of shape (n_pos+n_neg,), real-valued).
    """
    acc_clamped = np.clip(acc, 1e-6, 1 - 1e-6)
    d = 2.0 * norm.ppf(acc_clamped)  # separation
    mu_pos, mu_neg = +d / 2.0, -d / 2.0
    # Unit variance for both classes
    pos_scores = rng.normal(loc=mu_pos, scale=1.0, size=n_pos)
    neg_scores = rng.normal(loc=mu_neg, scale=1.0, size=n_neg)
    scores = np.concatenate([pos_scores, neg_scores], axis=0)
    y_true = np.concatenate([np.ones(n_pos, dtype=int), np.zeros(n_neg, dtype=int)], axis=0)
    return y_true, scores, d

# Build ROC curves per model
curves = []
for name, acc in observed.items():
    y_true, scores, d = synthesize_scores_from_acc(acc, N_POS, N_NEG, rng)
    fpr, tpr, thr = roc_curve(y_true, scores)
    roc_auc = auc(fpr, tpr)
    curves.append({
        "name": name,
        "acc_target": acc,
        "d_separation": float(d),
        "fpr": fpr,
        "tpr": tpr,
        "auc": float(roc_auc),
        "n_pos": int(N_POS),
        "n_neg": int(N_NEG),
    })

# Plot ROC comparison
plt.figure(figsize=(7.5, 6))
for c in curves:
    plt.plot(c["fpr"], c["tpr"], lw=2, label=f"{c['name']} (AUC={c['auc']:.3f}, acc≈{c['acc_target']:.2f})")
plt.plot([0, 1], [0, 1], "k--", lw=1, label="Chance")
plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Comparison (synthetic, calibrated to observed accuracies)")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
fig_path = OUT_DIR / "roc_synthetic_from_experiments.png"
plt.savefig(fig_path, dpi=200)
plt.close()
print(f"Saved ROC figure → {fig_path}")

# Save metrics table
df_metrics = pd.DataFrame([{
    "model": c["name"],
    "target_acc": c["acc_target"],
    "AUC": c["auc"],
    "d_separation": c["d_separation"],
    "n_pos": c["n_pos"],
    "n_neg": c["n_neg"],
} for c in curves]).sort_values("AUC", ascending=False)
csv_path = OUT_DIR / "roc_metrics.csv"
df_metrics.to_csv(csv_path, index=False)
print(f"Saved metrics → {csv_path}")

# Optional: print table
print(df_metrics.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
